### Using a Remote MCP Server as Tools

In this section we connect to the **Open Web Search MCP server** deployed on Azure Container Apps (see `aca_mcp_server.tf`) using [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters). The adapter discovers the server's tools automatically and makes them available as LangChain tools.

Github page for Open Web Search: https://github.com/Aas-ee/open-webSearch

In [1]:
%pip install langchain langgraph langchain-openai langchain-mcp-adapters

Note: you may need to restart the kernel to use updated packages.


In [1]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

aca_mcp_server_open_web_search_fqdn = ! terraform -chdir=infra output -raw aca_mcp_server_open_web_search_fqdn
aca_mcp_server_open_web_search_fqdn = aca_mcp_server_open_web_search_fqdn.n
print("MCP Server Endpoint:", aca_mcp_server_open_web_search_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4
MCP Server Endpoint: aca-mcp-server-open-web-search.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Connect to the LLM endpoint hosted on ACA with GPU
# model = ChatOpenAI(
#     base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
#     api_key="EMPTY",
#     model="google/gemma-4-31B-it",
#     streaming=True,
#     max_completion_tokens= 4096 # 8736 # 131072 # 512
# )

# Connect to the LLM endpoint hosted on Foundry
model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name_chatgpt,
    streaming=True,
    # max_completion_tokens=512
)

In [3]:
response = model.stream([HumanMessage(content="Tell me briefly about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI. I can help with writing, coding, explanations, brainstorming, summarizing, and answering questions across many topics.

I don’t have personal experiences or feelings, but I can simulate conversation and adapt to what you need. If you want, I can be concise, detailed, technical, casual, or more creative.

### Connect to the Remote MCP Server and Discover Tools

Use `MultiServerMCPClient` to connect to the MCP server over **Streamable HTTP** transport. The client automatically discovers all tools the server exposes.

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
mcp_client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_open_web_search_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


In [5]:
# Invoke a specific tool
import json

# Find the tool called "search" by name
search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

result = await search_tool.ainvoke({"query": "What are the latest LLM models?", "limit": 20, "engines": ["duckduckgo"]})

# result is already a list — extract the "text" field and parse it
data = json.loads(result[0]["text"])
print(json.dumps(data, indent=2))

{
  "query": "What are the latest LLM models?",
  "engines": [
    "duckduckgo"
  ],
  "totalResults": 20,
  "results": [
    {
      "title": "LLM Leaderboard 2026: Compare 300+ Top AI Models by Intelligence, Speed ...",
      "url": "https://llm-stats.com/",
      "description": "<b>The</b> <b>LLM</b> Leaderboard \u2014 independent ranking of GPT, Claude, Gemini, Llama, DeepSeek and 300+ AI <b>models</b> by intelligence, speed and price. Composite <b>LLM</b> Stats Score updated continuously from public benchmarks and live API metrics.",
      "source": "llm-stats.com",
      "engine": "duckduckgo"
    },
    {
      "title": "Best LLM Leaderboard 2026 | AI Model Rankings, Benchmarks &amp; Pricing",
      "url": "https://onyx.app/llm-leaderboard",
      "description": "<b>The</b> definitive <b>LLM</b> leaderboard \u2014 ranking the best AI <b>models</b> including Claude, GPT, Gemini, DeepSeek, Llama, and more across coding, reasoning, math, agentic, and chat benchmarks. Compare <b>LLM

### Run the Agent with MCP Tools

The agent will use the remote web search MCP tool to answer questions that require live information from the internet.

In [6]:
import httpx
from langchain.tools import tool
from markdownify import markdownify

@tool
def fetch_webpage_content(url: str, timeout: float = 10.0) -> str:
    """Fetch webpage and convert HTML to markdown."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        response = httpx.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        return markdownify(response.text)
    except Exception as e:
        return f"Error fetching {url}: {e!s}"

In [7]:
from IPython.display import Markdown, display

search_tool = next(t for t in mcp_tools_web_search if t.name == "search")

agent_with_mcp = create_agent(model=model, tools=[search_tool, fetch_webpage_content])

last_message = None

async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content=
    """
        What are the latest open LLM models in 2026 ?
        Compare their capabilities and limitations.
        Precise how many parameters each model has and the size of their input context window and the date of release.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    """
    # """
    #     What are the latest news and updates to AI agents from Microsoft, Google, AWS, Anthropic and OpenAI ?
    #     Search the web for 50 result pages. Fetch webpages.
    #     Cite your references in the response at the end.
    # """
    )]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    last_message = step["messages"][-1]

# show the response content as markdown
content = getattr(last_message, "content", "")
if isinstance(content, list):
    markdown_text = "\n".join(
        part.get("text", "") if isinstance(part, dict) else str(part)
        for part in content
    )
else:
    markdown_text = str(content)

display(Markdown(markdown_text))

================================ Human Message =================================


        What are the latest open LLM models in 2026 ?
        Compare their capabilities and limitations.
        Precise how many parameters each model has and the size of their input context window and the date of release.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    
================================== Ai Message ==================================
Tool Calls:
  search (call_NLDl4SomkiMN60aNdigBFms7)
 Call ID: call_NLDl4SomkiMN60aNdigBFms7
  Args:
    query: latest open LLM models 2026 parameters context window release date open weights model card 2025 2026
    limit: 50
    searchMode: auto
    engines: ['startpage', 'duckduckgo']
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "latest open LLM models 2026 parameters context window release d

I searched 50 web result pages and fetched a set of primary and secondary sources. Based on those fetched pages, here is a practical 2026 snapshot of the most relevant **open / open-weight LLMs**.

## First: a precision note on “open”
In 2026, many people say “open-source LLM,” but several leading models are more accurately **open-weight** rather than fully open-source. I’ll still compare them because that is what the ecosystem usually means by “open LLMs.” I’ll call out licenses where relevant.

---

# Latest notable open LLM models in 2026

## Summary table

| Model | Release date | Parameters | Context window | Main strengths | Main limitations |
|---|---:|---:|---:|---|---|
| **DeepSeek-V4-Pro** | 2026 | **1.6T total / 49B active** | **1M** | Frontier open model; excellent reasoning, coding, long-context | Very large; self-hosting is hard/expensive |
| **DeepSeek-V4-Flash** | 2026 | **284B total / 13B active** | **1M** | Faster/cheaper V4 variant; strong long-context | Lower ceiling than Pro |
| **Qwen3.6-35B-A3B** | 2026-04-16 | **35B total / 3B active** | **262,144** | Very efficient coding/agentic model | Smaller knowledge ceiling than flagship open MoEs |
| **Qwen3.6-27B** | 2026-04-22 | **27B dense** | **262,144** | Strong dense model for coding and local-ish use | No MoE efficiency advantage at scale |
| **Qwen3.5-397B-A17B** | 2026-02-16 | **397B total / 17B active** | **262,144** | Strong frontier open family; multilingual, multimodal family | Heavy infra needs |
| **Qwen3.5-122B-A10B** | 2026-02-24 | **122B total / 10B active** | **262,144** | Better deployability than 397B while still strong | Still expensive to self-host |
| **Qwen3.5-35B-A3B** | 2026-02-24 | **35B total / 3B active** | **262,144** | Good latency/quality tradeoff | Lower absolute performance |
| **GLM-5** | 2026-02-12 | **744B total / 40B active** | **200K input / up to 128K output** | Long-horizon coding/agentic workflows; huge output budget | Very demanding hardware |
| **MiniMax-M2.5** | 2026-02-12 | **229B total / 10B active** | **196,608** | Strong SWE / coding productivity; efficient active params | More uneven outside structured coding tasks |
| **Mistral Medium 3.5** | 2026-04 | *(parameter count not disclosed in fetched source)* | **256K** | Frontier-class multimodal open model for coding/agents | Missing public parameter count in fetched docs |
| **Mistral Small 4** | 2026-03 | *(parameter count not disclosed in fetched source)* | **256K** | Efficient hybrid instruct/reasoning/coding model | Parameter count not in fetched overview |
| **Gemma 4 26B A4B** | 2026 | **25.2B total / 3.8B active** | **256K** | Strong local/deployable model; Apache 2.0 | Lower frontier ceiling than giant MoEs |
| **Llama 4 Scout** | 2025-04-05 | **109B total / 17B active** | **10M** | By far the standout long-context open model | License is more restrictive than Apache/MIT |
| **Llama 4 Maverick** | 2025-04-05 | **400B total / 17B active** | *(not clearly stated in fetched page; Scout is the 10M focus)* | Strong multimodal, coding, reasoning | Very large; license restrictions |
| **GPT-OSS-120B** | 2025 | **117B total / 5.1B active** | **131K** | Good reasoning/tool-use, efficient serving | Lower ceiling than newest 2026 frontier open models |
| **GPT-OSS-20B** | 2025 | **21B total / 3.6B active** | **131K** | Small, practical open model | Not frontier-level |

---

# Best open models in 2026, compared

## 1) DeepSeek-V4-Pro
- **Release:** 2026
- **Parameters:** **1.6 trillion total, 49B active**
- **Context:** **1M tokens**
- **License:** MIT
- **Positioning:** strongest fetched open-weight frontier model in raw scale

### Capabilities
- Excellent on reasoning, coding, agentic tasks, and long-context work.
- Supports multiple reasoning effort modes.
- Strong benchmark profile against closed models in the fetched model card.

### Limitations
- Extremely difficult to self-host economically.
- Real-world latency can be substantial.
- Best for teams with serious infrastructure or hosted inference.

### Best for
- Frontier open deployments
- Advanced coding agents
- Massive-document or repo-scale reasoning

---

## 2) DeepSeek-V4-Flash
- **Release:** 2026
- **Parameters:** **284B total, 13B active**
- **Context:** **1M**
- **License:** MIT

### Capabilities
- Same million-token design philosophy as Pro.
- Much more efficiency-oriented.
- Better fit for production throughput-sensitive use.

### Limitations
- Less capable than V4-Pro on hardest tasks.
- Still not “lightweight.”

### Best for
- Cost-sensitive long-context systems
- Faster production inference than Pro

---

## 3) Qwen3.6 family
### Qwen3.6-35B-A3B
- **Release:** **2026-04-16**
- **Parameters:** **35B total, 3B active**
- **Context:** **262,144**

### Qwen3.6-27B
- **Release:** **2026-04-22**
- **Parameters:** **27B dense**
- **Context:** **262,144**

### Capabilities
- Strong focus on **agentic coding** and repository-level reasoning.
- Much more practical than trillion-class models.
- Apache 2.0 licensing is a big advantage for commercial adoption.

### Limitations
- Lower knowledge/reasoning ceiling than DeepSeek-V4-Pro or GLM-5.
- Qwen3.6 docs fetched do not give a full benchmark matrix in one place.

### Best for
- Coding assistants
- Self-hosting with more realistic hardware
- Commercial products needing permissive licensing

---

## 4) Qwen3.5 family
### Key fetched releases
- **Qwen3.5-397B-A17B** — **2026-02-16**, **397B total / 17B active**, **262,144**
- **Qwen3.5-122B-A10B** — **2026-02-24**, **122B / 10B active**, **262,144**
- **Qwen3.5-35B-A3B** — **2026-02-24**, **35B / 3B active**, **262,144**

### Capabilities
- Broad family from deployable to frontier-ish.
- Strong multilingual and multimodal direction.
- Good balance of ecosystem, open availability, and scale.

### Limitations
- Bigger variants still need major infra.
- The newest Qwen3.6 variants are often the more practical choice unless you need flagship scale.

### Best for
- Organizations wanting a full scalable open family
- Multilingual commercial deployments

---

## 5) GLM-5
- **Release:** **2026-02-12**
- **Parameters:** **744B total / 40B active**
- **Context:** **200K input**, up to **128K output**
- **License:** MIT weights / Apache 2.0 code

### Capabilities
- Particularly notable for **very large output length**, which matters for code generation and long reports.
- Strong on systems engineering, coding agents, long-horizon tasks.
- More “agent-work” oriented than casual chat.

### Limitations
- Very heavy to run locally.
- Lower context than DeepSeek-V4 and Llama 4 Scout.
- Operational complexity remains high.

### Best for
- Large code generation
- Long-form output workflows
- Agentic systems that must generate a lot, not just read a lot

---

## 6) MiniMax-M2.5
- **Release:** **2026-02-12**
- **Parameters:** **229B total / 10B active**
- **Context:** **196,608**
- **License:** Modified MIT

### Capabilities
- Strong SWE-bench style performance in fetched secondary sources.
- Very efficient active-parameter ratio.
- Good for structured engineering and “spec-first” coding workflows.

### Limitations
- Can be weaker on open-ended/general tasks.
- Less universally adopted ecosystem than Qwen/DeepSeek/Llama.

### Best for
- Software engineering workflows
- Cost-aware coding deployments

---

## 7) Mistral Medium 3.5 and Mistral Small 4
From the fetched Mistral docs:
- **Mistral Medium 3.5** — **release: 2026-04**, **context: 256K**
- **Mistral Small 4** — **release: 2026-03**, **context: 256K**

### Capabilities
- Mistral positions Medium 3.5 as a frontier-class multimodal model optimized for coding and agentic uses.
- Small 4 is presented as a hybrid instruct/reasoning/coding model.

### Limitations
- In the fetched official overview page, **parameter counts were not disclosed**, so I cannot state them precisely without guessing.
- Because your request asked for precise parameters, I have to mark them as **not specified in the fetched source**.

### Best for
- Teams already in the Mistral stack
- Efficient multilingual and coding-focused deployments

---

## 8) Gemma 4 26B A4B
- **Release:** 2026
- **Parameters:** **25.2B total / 3.8B active**
- **Context:** **256K**
- **License:** Apache 2.0

### Capabilities
- One of the best practical open models for local or moderate-scale deployment.
- Very favorable capability-to-hardware ratio.
- Permissive licensing is excellent for commercial work.

### Limitations
- Not a frontier giant; it will lose to DeepSeek-V4-Pro / GLM-5 / big Qwen on hardest tasks.
- Better seen as a pragmatic model than the absolute top benchmark leader.

### Best for
- Local/private deployments
- SME products
- Cost-sensitive commercial use

---

## 9) Llama 4 Scout and Maverick
### Llama 4 Scout
- **Release:** **2025-04-05**
- **Parameters:** **109B total / 17B active**
- **Context:** **10M**

### Llama 4 Maverick
- **Release:** **2025-04-05**
- **Parameters:** **400B total / 17B active**
- **Context:** Meta’s fetched blog emphasizes Scout’s **10M** context; Maverick is described as large multimodal open-weight but the exact input window is not clearly isolated in the fetched text.

### Capabilities
- **Scout** is the open-model long-context outlier: **10 million tokens**.
- Maverick is a strong multimodal and coding/reasoning model.
- Good ecosystem support and widespread deployment availability.

### Limitations
- License is not as clean as Apache 2.0 or MIT.
- In 2026, newer Qwen / DeepSeek / GLM families often look stronger on coding/reasoning frontiers.
- Maverick/Scout are still very large.

### Best for
- Extreme long-context tasks: Scout
- Meta ecosystem users
- Multimodal open-weight experimentation

---

## 10) GPT-OSS models
### GPT-OSS-120B
- **Release:** 2025
- **Parameters:** **117B total / 5.1B active**
- **Context:** **131K**

### GPT-OSS-20B
- **Release:** 2025
- **Parameters:** **21B total / 3.6B active**
- **Context:** **131K**

### Capabilities
- Efficient and practical open models.
- Useful for tool use and smaller deployments.
- Apache 2.0 license is a major plus.

### Limitations
- Not at the 2026 open frontier anymore.
- Smaller context than many current leaders.
- Less suited for million-token or ultra-long-horizon tasks.

### Best for
- Smaller production systems
- Developers wanting OpenAI-origin open-weight options

---

# Which open model is best in 2026?

## Best overall frontier open model
**DeepSeek-V4-Pro**
- Best combination of scale, long context, and strong benchmark profile in fetched primary sources.

## Best for coding agents
**GLM-5**, **Qwen3.6-35B-A3B**, **DeepSeek-V4-Pro**
- GLM-5 stands out for huge output budgets.
- Qwen3.6-35B-A3B stands out for efficiency and practicality.
- DeepSeek-V4-Pro stands out for top-end capability.

## Best practical commercial choice
**Qwen3.6 / Qwen3.5 / Gemma 4**
- Mostly because of permissive licensing and more realistic deployment profiles.

## Best long-context model
**Llama 4 Scout**
- **10M tokens** is unmatched in the fetched primary sources.

## Best local-ish open model
**Gemma 4 26B A4B**
- Best balance of quality, context, and active-parameter efficiency from the fetched sources.

---

# Capability comparison by dimension

## Reasoning
Top tier:
- DeepSeek-V4-Pro
- GLM-5
- Qwen3.5-397B-A17B
- Qwen3.6 family (good practical reasoning)
- MiniMax-M2.5

Limitation across the board:
- Open models still often trail the best closed models on absolute reliability, calibration, and polished assistant behavior.

## Coding / software engineering
Top tier:
- GLM-5
- DeepSeek-V4-Pro
- Qwen3.6-35B-A3B
- MiniMax-M2.5

Limitations:
- Large models are expensive to run.
- Smaller efficient models may be better for iterative IDE usage than the raw top models.

## Long context
Top tier:
- Llama 4 Scout — **10M**
- DeepSeek-V4-Pro / Flash — **1M**
- Some Qwen variants can extend high, but official fetched GitHub pages clearly show **262,144** for Qwen3.5/3.6 serving examples.

Limitations:
- Advertised context ≠ reliable effective reasoning at full length.
- Latency and memory costs rise sharply.

## Licensing / commercial deployability
Best:
- Apache 2.0: Qwen, Gemma, GPT-OSS
- MIT: DeepSeek, parts of GLM stack

More restrictive:
- Llama community license
- Modified MIT variants like Kimi/MiniMax
- Some Mistral variants may have special terms depending on model

---

# Short recommendations

## If you want one model to test first
- **DeepSeek-V4-Pro** for maximum capability
- **Qwen3.6-35B-A3B** for practical coding + deployment balance
- **Gemma 4 26B A4B** for local/private use
- **Llama 4 Scout** if long context is the main requirement

## If you are a startup shipping commercially
Start with:
- **Qwen3.6**
- **Gemma 4**
- optionally **DeepSeek-V4-Flash/Pro** via hosted inference

## If you want self-hosting with sane hardware
Start with:
- **Gemma 4 26B A4B**
- **Qwen3.6-27B**
- **Qwen3.6-35B-A3B** if you can handle MoE serving

---

# Important caveat
You asked for the “latest open LLM models in 2026.” Some top 2026 models appearing in search results were discussed mainly on secondary sites rather than fetched official model cards. I prioritized **official fetched sources** whenever available. Where an official fetched source did **not** provide a precise parameter count, I did **not invent one**.

---

# References

Primary / official sources fetched:
1. Meta AI — *The Llama 4 herd: The beginning of a new era of natively multimodal AI innovation*  
   https://ai.meta.com/blog/llama-4-multimodal-intelligence/

2. Qwen GitHub — *Qwen3*  
   https://github.com/QwenLM/Qwen3

3. Qwen GitHub — *Qwen3.6*  
   https://github.com/QwenLM/Qwen3.6

4. Mistral Docs — *Models Overview*  
   https://docs.mistral.ai/models/overview

5. Hugging Face — *deepseek-ai/DeepSeek-V4-Pro*  
   https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro

6. IBM Think — *A list of large language models (LLMs)*  
   https://www.ibm.com/think/topics/large-language-models-list

Secondary / comparative sources fetched:
7. Fireworks AI — *Best Open Source LLMs in 2026: We Reviewed 7 Models*  
   https://fireworks.ai/blog/best-open-source-llms

8. LLM Stats — *LLM Leaderboard 2026*  
   https://llm-stats.com/

9. Artificial Analysis — *LLM Leaderboard*  
   https://artificialanalysis.ai/leaderboards/models

10. Hugging Face community article — *Best Open-Source LLM Models in 2026*  
   https://huggingface.co/blog/daya-shankar/open-source-llms

11. Clarifai — *Top 10 Open-source Reasoning Models in 2026*  
   https://www.clarifai.com/blog/top-10-open-source-reasoning-models-in-2026

If you want, I can turn this into:
1. a **clean CSV/Markdown comparison table**, or  
2. a **ranked top-10 with only official-source-confirmed specs**.

## Working with Sub-Agents

A main agent coordinates subagents as tools. All routing passes through the main agent.

In the subagents architecture, a central main agent (often referred to as a supervisor) coordinates subagents by calling them as tools. The main agent decides which subagent to invoke, what input to provide, and how to combine results. Subagents are stateless—they don’t remember past interactions, with all conversation memory maintained by the main agent. This provides context isolation: each subagent invocation works in a clean context window, preventing context bloat in the main conversation.

![](./images/sub_agents.png)


### When to use

Use the subagents pattern when you have multiple distinct domains (e.g., calendar, email, CRM, database), subagents don’t need to converse directly with users, or you want centralized workflow control. For simpler cases with just a few tools, use a single agent.

### Performance comparison

Different patterns have different performance characteristics. Understanding these tradeoffs helps you choose the right pattern for your latency and cost requirements.
Key metrics:
Model calls: Number of LLM invocations. More calls = higher latency (especially if sequential) and higher per-request API costs.
Tokens processed: Total context window usage across all calls. More tokens = higher processing costs and potential context limits.

In [9]:
from langchain.tools import tool
from langchain.agents import create_agent

# Create a subagent
subagent = create_agent(model=model, tools=[search_tool, fetch_webpage_content])
# subagent = create_agent(model="google_genai:gemini-3.1-pro-preview", tools=[...])

# Wrap it as an async tool so it stays in the async execution path
@tool("research", description="Research a topic and return findings")
async def call_sub_agent(query: str):
    result = await subagent.ainvoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].content

# Main agent with subagent as a tool
main_agent = create_agent(model=model, 
                          tools=[call_sub_agent],     
                          system_prompt=(""""
    You coordinate specialized sub-agent specialized in web search. 
    Use the task tool to delegate work.
    Delegate fetch web pages task to the sub agent tool and let it fetch the content of that page and report it back to the main agent.
"""))


In [11]:
last_message = None

async for step in main_agent.astream(
    {"messages": [HumanMessage(content=
    """
        What are the latest open LLM models in 2026 ?
        Compare their capabilities and limitations.
        Precise how many parameters each model has and the size of their input context window and the date of release.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    """
    # """
    #     What are the latest news and updates to AI agents from Microsoft, Google, AWS, Anthropic and OpenAI ?
    #     Search the web for 50 result pages. Fetch webpages.
    #     Cite your references in the response at the end.
    # """
    )]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    last_message = step["messages"][-1]

# show the response content as markdown
content = getattr(last_message, "content", "")
if isinstance(content, list):
    markdown_text = "\n".join(
        part.get("text", "") if isinstance(part, dict) else str(part)
        for part in content
    )
else:
    markdown_text = str(content)

display(Markdown(markdown_text))

================================ Human Message =================================


        What are the latest open LLM models in 2026 ?
        Compare their capabilities and limitations.
        Precise how many parameters each model has and the size of their input context window and the date of release.
        Search the web for 50 result pages. Fetch webpages.
        Cite your references in the response at the end.
    
================================== Ai Message ==================================
Tool Calls:
  research (call_1dJrgad78ru92hvJ4Iwuyalw)
 Call ID: call_1dJrgad78ru92hvJ4Iwuyalw
  Args:
    query: latest open LLM models 2026 parameters context window release date open weights 2025 2026 50 results
  research (call_ei5uyqpVC5QBs6Eo8SwiQgHa)
 Call ID: call_ei5uyqpVC5QBs6Eo8SwiQgHa
  Args:
    query: 2026 open source LLM models comparison parameters context window release date fetch sources
  research (call_0VdIunuxAXmAKOFmmpnrz3Z6)
 Call ID: call_0VdIunuxAXmAKOFmmpnrz3

Below is a web-researched snapshot of the most relevant **open-weight / open-available LLMs current going into 2026**.

## Important framing
- There are **not many clearly “released in 2026” flagship open models yet** from official sources.
- So the best answer to “latest open LLM models in 2026” is really: **the newest major open models available as of 2025–2026**.
- I use **official pages/model cards/repos where possible**.
- I also distinguish:
  - **Open-weight**: model weights available, but not necessarily full training data or permissive OSI-style openness.
  - **Fully open**: weights + code + recipe/data transparency.  
  In this list, **OLMo** is the strongest example of “fully open.”

---

# Short answer

As of 2026, the most important recent open/open-weight LLMs are:

- **Meta Llama 4 Scout**
- **Meta Llama 4 Maverick**
- **Qwen3 family**
- **DeepSeek-V3**
- **Kimi K2**
- **GLM-4.5**
- **Mistral Small 3.1**
- **Gemma 3**
- **OLMo 2 32B**

These models differ mainly on:
- **raw capability**
- **reasoning**
- **multimodality**
- **context length**
- **deployment cost**
- **license openness**
- **how “open” they truly are**

---

# Comparison table

| Model | Release date | Parameters | Context window | Strengths | Main limitations |
|---|---:|---:|---:|---|---|
| **Llama 4 Scout** | 2025-04-05 | **109B total, 17B active, 16 experts** | **10M** | Exceptional long context, multimodal, strong general model | MoE complexity, license not fully open-source, real use of 10M context is costly |
| **Llama 4 Maverick** | 2025-04-05 | **400B total, 17B active, 128 experts** | not clearly stated on fetched official page | Strong flagship open-weight Meta model, multimodal | Heavy serving complexity, less transparent than fully open models |
| **Qwen3-235B-A22B** | 2025-04-29 | **235B total, 22B active** | **128K** | Very strong reasoning/general performance, multilingual, MoE efficiency | Large deployment footprint, license/openness not fully equivalent to Apache-style OSS |
| **Qwen3-32B** | 2025-04-29 | **32B dense** | **128K** | Strong dense model, easier to run than giant MoE models | Lower ceiling than top MoE flagships |
| **DeepSeek-V3** | 2024-12-26 | **671B total, 37B active** | **128K** | Excellent capability/efficiency tradeoff for a frontier-class MoE model | Operational complexity, official support caveats, very large infrastructure needs |
| **Kimi K2** | 2025 (official repo/report year) | **1T total, 32B active** | **128K** | Very large open MoE model, likely frontier-class | Very demanding to serve; less mature ecosystem than older families |
| **GLM-4.5** | 2025-07-28 | **355B total, 32B active** | **128K** | Strong hybrid reasoning model, open repo | Large hardware requirements; ecosystem smaller than Llama/Qwen |
| **Mistral Small 3.1** | 2025-03-17 | **24B** | **128K** | Strong compact model, multimodal, Apache 2.0, practical deployment | Smaller than top frontier open models; less absolute capability ceiling |
| **Gemma 3 27B** | 2025-03-12 | **27B** | **128K** | Efficient, practical, good small-to-mid-size option | Generally below the biggest MoE leaders in top-end capability |
| **OLMo 2 32B** | 2025-03-13 | **32B** | not clearly stated on fetched official page | Most genuinely open/reproducible, strong research value | Usually not the top absolute performer versus biggest industrial MoE models |

---

# Detailed comparison

## 1) Meta Llama 4 Scout
- **Release date:** 2025-04-05
- **Parameters:** **109B total**, **17B active**, **16 experts**
- **Context window:** **10 million tokens**
- **Type:** multimodal MoE open-weight model

### Capabilities
- One of the most notable open-weight releases because of its **extreme long-context support**
- Good for:
  - large retrieval contexts
  - long-document analysis
  - agent workflows with persistent memory
  - multimodal tasks

### Limitations
- The **10M-token context** is impressive, but in practice:
  - expensive
  - memory intensive
  - not equally efficient in all deployments
- Meta’s models are **open-weight**, not fully open-source in the strongest sense
- MoE routing makes deployment more complex than smaller dense models

---

## 2) Meta Llama 4 Maverick
- **Release date:** 2025-04-05
- **Parameters:** **400B total**, **17B active**, **128 experts**
- **Context window:** not clearly stated on the fetched official Meta page
- **Type:** multimodal MoE open-weight model

### Capabilities
- Positioned as a stronger flagship sibling to Scout
- Better suited for:
  - advanced general chat
  - multimodal tasks
  - stronger reasoning and instruction following

### Limitations
- Large total parameter count means:
  - higher infrastructure complexity
  - more difficult serving/optimization
- Context size was not clearly extractable from the official page fetch I used, so I won’t invent it

---

## 3) Qwen3 family
- **Release date:** 2025-04-29
- **Models:**  
  - **235B-A22B**
  - **30B-A3B**
  - **32B**
  - **14B**
  - **8B**
  - **4B**
  - **1.7B**
  - **0.6B**

### Parameter counts and context
- **Qwen3-235B-A22B**: **235B total / 22B active**, **128K**
- **Qwen3-30B-A3B**: **30B total / 3B active**, **128K**
- **Qwen3-32B**: **32B dense**, **128K**
- **Qwen3-14B**: **14B**, **128K**
- **Qwen3-8B**: **8B**, **128K**
- **Qwen3-4B**: **4B**, **32K**
- **Qwen3-1.7B**: **1.7B**, **32K**
- **Qwen3-0.6B**: **0.6B**, **32K**

### Capabilities
- One of the strongest and broadest open families available
- Strong at:
  - multilingual tasks
  - reasoning
  - coding
  - instruction following
- Offers both:
  - **frontier-style large MoE**
  - **smaller practical models**

### Limitations
- Larger Qwen3 models still need serious infrastructure
- “Open” is mostly **open-weight**, not fully open-training-data transparency
- Performance depends heavily on which variant you choose

---

## 4) DeepSeek-V3
- **Release date:** **2024-12-26**
- **Parameters:** **671B total / 37B active**
- **Context window:** **128K**
- **Type:** MoE open model

### Capabilities
- Among the most influential recent open models
- Strong on:
  - reasoning
  - coding
  - general assistant tasks
  - efficiency relative to total scale thanks to MoE

### Limitations
- Very large operational footprint
- Official repo notes ecosystem caveats such as tooling support
- Best suited for well-resourced deployments, not casual local use

---

## 5) Kimi K2
- **Release:** 2025 official repo/report
- **Parameters:** **1T total / 32B active**
- **Context window:** **128K**
- **Type:** very large MoE open-weight model

### Capabilities
- Extremely large model by total parameter count
- Promising for top-tier reasoning/general capability

### Limitations
- Inference and serving are difficult
- Smaller surrounding ecosystem than Llama/Qwen
- Practical access may lag behind more established open model families

---

## 6) GLM-4.5
- **Release date:** **2025-07-28**
- **Parameters:** **355B total / 32B active**
- **Context window:** **128K**
- **Type:** hybrid reasoning MoE model

### Capabilities
- Strong reasoning-oriented open model
- Supports “thinking/non-thinking” style modes
- Competitive for advanced assistant tasks

### Limitations
- Hardware requirements are substantial
- Community/tooling ecosystem is smaller than top Western and Alibaba model families
- Less broadly adopted in open tooling stacks than Llama/Qwen

---

## 7) Mistral Small 3.1
- **Release date:** **2025-03-17**
- **Parameters:** **24B**
- **Context window:** **128K**
- **License:** **Apache 2.0**

### Capabilities
- One of the best practical medium-size open models
- Strong for:
  - enterprise deployment
  - local/private serving
  - fine-tuning
  - multimodal use
- Much easier to deploy than the giant MoE flagships

### Limitations
- Lower absolute top-end capability than the largest Qwen/DeepSeek/Llama/Kimi systems
- Better seen as a **very strong practical model**, not the strongest frontier open model

---

## 8) Gemma 3
- **Release date:** **2025-03-12**
- **Sizes:** **1B, 4B, 12B, 27B**
- **Context:**
  - **1B:** **32K**
  - **4B / 12B / 27B:** **128K**

### Capabilities
- Very good efficiency family
- Good for:
  - smaller hardware
  - fine-tuning
  - compact deployment
  - research and experimentation

### Limitations
- The larger frontier MoE models still outperform it at the high end
- Best viewed as an efficient practical family rather than a top absolute-capability leader

---

## 9) OLMo 2 32B
- **Release date:** **2025-03-13**
- **Parameters:** **32B**
- **Context window:** not clearly stated on the fetched official blog page
- **Type:** fully open research-grade model

### Capabilities
- Strongest point is **true openness**
- Valuable for:
  - reproducible research
  - transparency
  - studying training methods
  - open science

### Limitations
- Usually not the strongest absolute performer against giant industrial MoE systems
- More important scientifically than as the single best production chatbot

---

# Which open model is “best” in 2026?

Depends on the goal:

## Best frontier open-weight models
- **Llama 4 Maverick**
- **Qwen3-235B-A22B**
- **DeepSeek-V3**
- **Kimi K2**
- **GLM-4.5**

## Best practical medium-size model
- **Mistral Small 3.1**

## Best efficient smaller family
- **Gemma 3**

## Best for long context
- **Llama 4 Scout** by a huge margin on paper (**10M tokens**)

## Best for openness/transparency
- **OLMo 2 32B**

## Best all-around family breadth
- **Qwen3**
because it spans tiny to frontier-scale variants.

---

# Main capability differences

## Reasoning
Strongest recent open reasoning-oriented models appear to be:
- **Qwen3 large variants**
- **GLM-4.5**
- **DeepSeek-V3**
- **Kimi K2**
- **Llama 4 Maverick**

## Coding
Particularly strong families:
- **Qwen**
- **DeepSeek**
- **GLM**
- **Mistral** can be practical and cost-effective

## Multilingual
Best-known strength:
- **Qwen3**
- **Mistral Small 3.1**
- **Llama 4**
- **Gemma 3**

## Long-context
- **Llama 4 Scout** is the standout on paper
- Most others cluster around **128K**

---

# Main limitations across open models

## 1) Open-weight is not the same as fully open-source
Most leading models here are **not fully open in data/training recipe**:
- Llama
- Qwen
- DeepSeek
- Gemma
- GLM
- Kimi

Only **OLMo** is notably close to full openness.

## 2) Huge MoE models are hard to deploy
Even if “active” parameters are moderate, these models often require:
- multiple GPUs
- high VRAM
- optimized serving stacks
- sophisticated routing/inference setups

## 3) Context window claims are not equal to practical performance
A model may support **128K** or even **10M**, but:
- quality can degrade over long sequences
- retrieval and attention costs rise sharply
- many real applications chunk context anyway

## 4) Benchmarks do not perfectly predict real-world use
A model that looks better on benchmark charts may still be worse for:
- latency
- cost
- tool use
- factual reliability
- domain-specific tasks

---

# Best ranked summary

If I had to rank the most relevant open/open-weight LLMs available entering 2026:

1. **Qwen3-235B-A22B**
2. **Llama 4 Maverick**
3. **DeepSeek-V3**
4. **Kimi K2**
5. **GLM-4.5**
6. **Llama 4 Scout**
7. **Mistral Small 3.1**
8. **Gemma 3 27B**
9. **OLMo 2 32B**

That ranking balances:
- capability
- relevance
- recency
- ecosystem importance

But if you prioritize **real deployment**, I’d move **Mistral Small 3.1** and **Gemma 3** higher.

---

# About the “50 result pages” request

I performed web research and gathered a broad result set around open LLMs, then used targeted fetching of official pages for the key models.  
The search result set included around 50 candidate pages, but for factual comparison I prioritized **official model cards, official blogs, and official repos** over secondary articles.

---

# References

Official sources used for the facts above:

1. Meta AI — **The Llama 4 herd**  
   https://ai.meta.com/blog/llama-4-multimodal-intelligence/

2. Qwen official blog — **Qwen3**  
   https://qwenlm.github.io/blog/qwen3/

3. DeepSeek official GitHub — **DeepSeek-V3**  
   https://github.com/deepseek-ai/DeepSeek-V3

4. Mistral AI official news — **Mistral Small 3.1**  
   https://mistral.ai/news/mistral-small-3-1

5. Mistral docs model card — **Mistral Small 3.1 (25.03)**  
   https://docs.mistral.ai/models/model-cards/mistral-small-3-1-25-03

6. Google Developers Blog — **Introducing Gemma 3**  
   https://developers.googleblog.com/en/introducing-gemma3/

7. Google official Gemma model card — **gemma-3-27b-it**  
   https://huggingface.co/google/gemma-3-27b-it

8. Allen Institute for AI — **OLMo 2 32B**  
   https://allenai.org/blog/olmo2-32b

9. Moonshot AI official GitHub — **Kimi K2**  
   https://github.com/MoonshotAI/Kimi-K2

10. Moonshot AI official page — **Kimi K2**  
   https://moonshotai.github.io/Kimi-K2/

11. Z.ai official repo — **GLM-4.5**  
   https://github.com/zai-org/GLM-4.5

12. Z.ai official blog — **GLM-4.5**  
   https://z.ai/blog/glm-4.5

13. Meta official model card — **Llama 3.3 70B Instruct**  
   https://huggingface.co/meta-llama/Llama-3.3-70B-Instruct

14. Qwen official model card — **Qwen2.5-72B**  
   https://huggingface.co/Qwen/Qwen2.5-72B

15. Qwen official model card — **Qwen2.5-72B-Instruct**  
   https://huggingface.co/Qwen/Qwen2.5-72B-Instruct

If you want, I can next convert this into a **clean CSV/Markdown table with columns for license, modality, dense vs MoE, total params, active params, and recommended hardware**.